# 02 - Task 1 Report Figures

This notebook aggregates training logs and test-set evaluation artifacts from `outputs/` and produces the figures and tables used in the Task 1 (EEG classification) section of the final report.

**Run order before executing this notebook**

1. Train each baseline:
   ```bash
   python scripts/train_classification.py --config configs/classification_mlp_baseline.yaml
   python scripts/train_classification.py --config configs/classification_cnn_baseline.yaml
   python scripts/train_classification.py --config configs/classification_eegnet_baseline.yaml
   ```
2. Evaluate each baseline on the test split (note the checkpoint path printed by the training script):
   ```bash
   python scripts/eval_classification.py \
     --config configs/classification_eegnet_baseline.yaml \
     --checkpoint outputs/checkpoints/eegnet_baseline-<TIMESTAMP>/best_model.pt
   ```
3. (Optional) Run EEGNet ablations and evaluate each the same way:
   ```bash
   python scripts/train_classification.py --config configs/classification_eegnet_norm_none.yaml
   python scripts/eval_classification.py \
     --config configs/classification_eegnet_norm_none.yaml \
     --checkpoint outputs/checkpoints/eegnet_baseline-<TIMESTAMP>/best_model.pt
   ```
4. Run this notebook top-to-bottom. All generated figures are saved under `reports/figures/`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300

# Walk up from the notebook's cwd until we find the project root.
_here = Path.cwd().resolve()
ROOT = _here
while ROOT != ROOT.parent:
    if (ROOT / "configs").exists() and (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

OUTPUTS = ROOT / "outputs"
LOGS = OUTPUTS / "logs"
FIGS_OUT = ROOT / "reports" / "figures"
FIGS_OUT.mkdir(parents=True, exist_ok=True)

print(f"Project root:      {ROOT}")
print(f"Outputs dir:       {OUTPUTS}")
print(f"Report figures to: {FIGS_OUT}")

In [ ]:
# Display name -> model_name prefix used by training runs (outputs/logs/<model_name>-<timestamp>/).
BASELINES = {
    "MLP": "mlp_baseline",
    "Temporal CNN": "cnn_baseline",
    "EEGNet": "eegnet_baseline",
}

# Eval tag for each baseline run. The eval script derives this from the config
# filename (with the `classification_` prefix stripped).
BASELINE_EVAL_TAGS = {
    "MLP": "mlp_baseline",
    "Temporal CNN": "cnn_baseline",
    "EEGNet": "eegnet_baseline",
}

# EEGNet ablations. Values are the eval tags produced by each ablation config.
ABLATIONS = {
    "EEGNet (norm=none)": "eegnet_norm_none",
    "EEGNet (dropout=0.5)": "eegnet_dropout05",
}

# The model we highlight in the Task 1 analysis (best baseline on validation).
BEST_DISPLAY_NAME = "EEGNet"
BEST_EVAL_TAG = BASELINE_EVAL_TAGS[BEST_DISPLAY_NAME]


def latest_training_dir(model_name: str):
    """Return the most recent outputs/logs/<model_name>-<timestamp> directory."""
    matches = sorted(LOGS.glob(f"{model_name}-*"))
    return matches[-1] if matches else None


def load_train_history(model_name: str):
    run_dir = latest_training_dir(model_name)
    if run_dir is None:
        return None
    path = run_dir / "train_history.json"
    if not path.exists():
        return None
    return pd.DataFrame(json.loads(path.read_text()))


def load_train_summary(model_name: str):
    run_dir = latest_training_dir(model_name)
    if run_dir is None:
        return None
    path = run_dir / "train_summary.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())


def load_test_metrics(eval_tag: str):
    path = LOGS / eval_tag / "test_metrics.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())


def load_confusion_matrix(eval_tag: str):
    csv_path = OUTPUTS / "figures" / eval_tag / "confusion_matrix.csv"
    if not csv_path.exists():
        return None
    return pd.read_csv(csv_path).to_numpy()


def load_class_names():
    mapping = json.loads((ROOT / "data/processed/class_mapping.json").read_text())
    return [name for name, _ in sorted(mapping.items(), key=lambda kv: kv[1])]


CLASS_NAMES = load_class_names()
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

## 1. Class distribution by split

In [ ]:
split_counts = {}
for split_name in ["train", "val", "test"]:
    split_path = ROOT / f"data/splits/{split_name}.csv"
    if not split_path.exists():
        continue
    df = pd.read_csv(split_path)
    split_counts[split_name] = (
        df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    )

dist_df = pd.DataFrame(split_counts)

fig, ax = plt.subplots(figsize=(10, 4.5))
dist_df.plot(kind="bar", ax=ax)
ax.set_title("Trials per class, by split")
ax.set_ylabel("# trials")
ax.set_xlabel("Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGS_OUT / "class_distribution_by_split.png")
plt.show()

dist_df

## 2. Training curves (baselines)

Loss and accuracy per epoch for each baseline, taken from the most recent training run for that model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax_loss, ax_acc = axes

found_any = False
for display_name, model_name in BASELINES.items():
    hist = load_train_history(model_name)
    if hist is None:
        print(f"[skip] no training history for {display_name} ({model_name})")
        continue
    found_any = True
    ax_loss.plot(hist["epoch"], hist["train_loss"], linestyle="--", alpha=0.6, label=f"{display_name} (train)")
    ax_loss.plot(hist["epoch"], hist["val_loss"], label=f"{display_name} (val)")
    ax_acc.plot(hist["epoch"], 100 * hist["train_acc"], linestyle="--", alpha=0.6, label=f"{display_name} (train)")
    ax_acc.plot(hist["epoch"], 100 * hist["val_acc"], label=f"{display_name} (val)")

ax_loss.set_title("Loss")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Cross-entropy loss")
ax_loss.legend(fontsize=8)

ax_acc.set_title("Accuracy")
ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Accuracy (%)")
ax_acc.axhline(5.0, color="gray", linestyle=":", label="Chance (5%)")
ax_acc.legend(fontsize=8)

plt.tight_layout()
if found_any:
    plt.savefig(FIGS_OUT / "training_curves.png")
plt.show()

## 3. Baseline results summary

Combines best validation accuracy (from training) with test-set accuracy (from `eval_classification.py`) for each baseline.

In [ ]:
rows = []
for display_name, model_name in BASELINES.items():
    summary = load_train_summary(model_name)
    history = load_train_history(model_name)
    eval_tag = BASELINE_EVAL_TAGS[display_name]
    test = load_test_metrics(eval_tag)

    row = {"Model": display_name}
    if summary is not None:
        row["Epochs"] = summary.get("epochs")
        row["Best epoch"] = summary.get("best_epoch")
        if summary.get("best_val_acc") is not None:
            row["Best val acc (%)"] = round(100 * float(summary["best_val_acc"]), 2)
    if history is not None and len(history):
        row["Final train acc (%)"] = round(100 * float(history["train_acc"].iloc[-1]), 2)
        row["Final val acc (%)"] = round(100 * float(history["val_acc"].iloc[-1]), 2)
    if test is not None:
        row["Test acc (%)"] = round(100 * float(test["accuracy"]), 2)
        row["Test loss"] = round(float(test["loss"]), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(FIGS_OUT / "task1_results_table.csv", index=False)
summary_df

## 4. Confusion matrix (best baseline)

Row-normalized, so entry (i, j) is P(predicted = j | true = i).

In [ ]:
cm = load_confusion_matrix(BEST_EVAL_TAG)
if cm is None:
    print(f"No confusion matrix found for eval tag '{BEST_EVAL_TAG}'. Run eval_classification.py first.")
else:
    row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
    cm_norm = cm / row_sums

    fig, ax = plt.subplots(figsize=(8.5, 7))
    sns.heatmap(
        cm_norm,
        cmap="Blues",
        vmin=0.0,
        vmax=1.0,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar_kws={"label": "P(pred | true)"},
        ax=ax,
    )
    ax.set_title(f"Confusion matrix (row-normalized) - {BEST_DISPLAY_NAME}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGS_OUT / f"confusion_matrix_{BEST_EVAL_TAG}.png")
    plt.show()

    # Top-10 most confused off-diagonal pairs
    off_diag = cm.copy()
    np.fill_diagonal(off_diag, 0)
    pairs = [
        (CLASS_NAMES[i], CLASS_NAMES[j], int(off_diag[i, j]))
        for i in range(len(CLASS_NAMES))
        for j in range(len(CLASS_NAMES))
    ]
    pairs.sort(key=lambda x: -x[2])
    confused_df = pd.DataFrame(pairs[:10], columns=["True", "Predicted", "Count"])
    print("Top-10 most confused (true -> predicted) pairs:")
    confused_df

## 5. Per-subject test accuracy (best baseline)

In [ ]:
test = load_test_metrics(BEST_EVAL_TAG)
if test is None or "per_subject_accuracy" not in test:
    print(f"No per-subject metrics found for eval tag '{BEST_EVAL_TAG}'. Run eval_classification.py first.")
else:
    per_subj = pd.Series(test["per_subject_accuracy"]).sort_index() * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    per_subj.plot(kind="bar", ax=ax, color="steelblue")
    ax.axhline(5.0, color="gray", linestyle=":", label="Chance (5%)")
    ax.axhline(per_subj.mean(), color="orange", linestyle="--", label=f"Mean ({per_subj.mean():.2f}%)")
    ax.set_title(f"Per-subject test accuracy - {BEST_DISPLAY_NAME}")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Subject")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(FIGS_OUT / f"per_subject_accuracy_{BEST_EVAL_TAG}.png")
    plt.show()

    per_subj.to_frame(name="Test acc (%)")

## 6. EEGNet ablation comparison

Compares the EEGNet baseline against single-factor ablation configs. Each row corresponds to one `configs/classification_eegnet_*.yaml` config, so at most one hyperparameter differs between adjacent rows.

In [ ]:
abl_rows = []

baseline_test = load_test_metrics(BEST_EVAL_TAG)
if baseline_test is not None:
    abl_rows.append({
        "Run": "EEGNet (baseline)",
        "Config": "classification_eegnet_baseline",
        "Test acc (%)": round(100 * float(baseline_test["accuracy"]), 2),
        "Test loss": round(float(baseline_test["loss"]), 4),
    })

for display_name, tag in ABLATIONS.items():
    test = load_test_metrics(tag)
    if test is None:
        print(f"[skip] no test metrics for {display_name} (tag '{tag}')")
        continue
    abl_rows.append({
        "Run": display_name,
        "Config": f"classification_{tag}",
        "Test acc (%)": round(100 * float(test["accuracy"]), 2),
        "Test loss": round(float(test["loss"]), 4),
    })

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(FIGS_OUT / "task1_ablation_table.csv", index=False)

if len(abl_df) >= 2:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.barplot(data=abl_df, x="Run", y="Test acc (%)", ax=ax, color="steelblue")
    ax.set_title("EEGNet ablations - test accuracy")
    ax.axhline(5.0, color="gray", linestyle=":", label="Chance (5%)")
    ax.legend()
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(FIGS_OUT / "task1_ablation_comparison.png")
    plt.show()

abl_df

## 7. Saved artifacts

Running this notebook produces the following files (skipped entries just mean the corresponding training/eval run has not been executed yet):

- `reports/figures/class_distribution_by_split.png`
- `reports/figures/training_curves.png`
- `reports/figures/task1_results_table.csv`
- `reports/figures/confusion_matrix_eegnet_baseline.png`
- `reports/figures/per_subject_accuracy_eegnet_baseline.png`
- `reports/figures/task1_ablation_table.csv`
- `reports/figures/task1_ablation_comparison.png`